In [0]:
dbutils.widgets.text("FileName", "")
file_name = dbutils.widgets.get("FileName")

raw_path = f"abfss://data@karansa2s.dfs.core.windows.net/raw/{file_name}"


In [0]:


storage_account = "karansa2s"
storage_key = "<KEY>"

spark.conf.set(
  f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
  storage_key
)


In [0]:
raw_path = "abfss://data@karansa2s.dfs.core.windows.net/raw/sales_2025-12-01.csv"

df = spark.read.format("csv").option('header', 'true').option('inferSchema', 'true').load(raw_path)

display(df)


bronze_path = "abfss://data@karansa2s.dfs.core.windows.net/bronze/sales_2025-12-01"
df.write.mode("overwrite").csv(bronze_path)



In [0]:
from pyspark.sql.functions import col

df_clean = df.filter(
    (col("SaleDate").isNotNull()) &
    (col("Amount") > 0)
)

df_reject = df.filter(
    (col("SaleDate").isNull()) |
    (col("Amount") <= 0)
)


In [0]:
from pyspark.sql.functions import regexp_replace, expr, substring, lit

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(StringType())
def Memail(x):
    f = x[0]   
    count = 0
    for i in x:
        if i == "@":
            break
        else:
            count += 1
    index = x[count:]
    for _ in range(count - 1):
        f += "*"
    return f + index

df_masked = df_clean.withColumn("Email", Memail("Email"))


silver_path = "abfss://data@karansa2s.dfs.core.windows.net/silver/sales_2025-12-01"

df_masked.write.mode("overwrite").csv(silver_path)

reject_path = "abfss://data@karansa2s.dfs.core.windows.net/reject/sales_2025-12-01"

df_reject.write.mode("overwrite").csv(reject_path, header=True)

